# Módulo 1: Exploración de Datos — Riesgo Operacional (CBK)

**Grupo 3** — Analítica Digital UCEMA 2026  
**Integrantes**: Ivan Charczuk, Robertino Barbuto, Sofia Martinez Luque

**Objetivo**: Explorar el dataset de órdenes enriquecido para entender la magnitud del riesgo por Chargebacks (CBK), identificar sus predictores y cuantificar el impacto en el margen de Olist.

## 1. Imports y Carga de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

os.makedirs('img', exist_ok=True)
os.makedirs('exports_tableau', exist_ok=True)

In [ ]:
# Ajustar ruta según donde tengas los archivos
df = pd.read_csv('olist_order_items_enriched.csv')

print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
df.head()

## 2. Estadísticas Descriptivas

In [ ]:
cols_analisis = ['price', 'profit', 'cbk', 'payment_installments',
                 'freight_value', 'discount', 'overhead_cost']
df[cols_analisis].describe().round(2)

In [ ]:
# Distribución de valores nulos
df.isnull().sum()[df.isnull().sum() > 0]

## 3. Tasa de CBK General

In [ ]:
cbk_rate = df['cbk'].mean() * 100
total_cbk = int(df['cbk'].sum())
total_ordenes = len(df)

# Pérdida real por CBK: -(overhead_cost + freight_value)
perdida_por_orden = df[df['cbk'] == 1].apply(
    lambda r: -(r['overhead_cost'] + r['freight_value']), axis=1
)
perdida_total = perdida_por_orden.sum()
perdida_promedio = perdida_por_orden.mean()

print(f'--- KPIs de CBK ---')
print(f'Total órdenes:          {total_ordenes:,}')
print(f'Órdenes con CBK:        {total_cbk:,} ({cbk_rate:.2f}%)')
print(f'Pérdida total por CBK:  ${perdida_total:,.2f}')
print(f'Pérdida promedio/CBK:   ${perdida_promedio:,.2f}')

## 4. Gráfico 1 — CBK Rate por Categoría de Producto

In [ ]:
cat_cbk = (
    df.groupby('product_category_name')['cbk']
    .agg(cbk_rate='mean', total='count')
    .reset_index()
)
cat_cbk = cat_cbk[cat_cbk['total'] > 100].copy()
cat_cbk['cbk_rate_pct'] = cat_cbk['cbk_rate'] * 100
cat_cbk = cat_cbk.sort_values('cbk_rate_pct', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(cat_cbk['product_category_name'], cat_cbk['cbk_rate_pct'],
               color=sns.color_palette('Reds_r', len(cat_cbk)))
ax.axvline(cbk_rate, color='navy', linestyle='--', linewidth=1.5, label=f'Promedio general ({cbk_rate:.1f}%)')
ax.set_xlabel('CBK Rate (%)')
ax.set_title('CBK Rate por Categoría de Producto\n(Top 15, mínimo 100 órdenes)', fontsize=13, fontweight='bold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('img/cbk_rate_categoria.png')
plt.show()

## 5. Gráfico 2 — CBK Rate por Número de Cuotas

In [ ]:
cuotas_cbk = (
    df.groupby('payment_installments')
    .agg(cbk_rate=('cbk', 'mean'), total=('cbk', 'count'))
    .reset_index()
)
cuotas_cbk['cbk_rate_pct'] = cuotas_cbk['cbk_rate'] * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(cuotas_cbk['payment_installments'].astype(str) + 'x',
       cuotas_cbk['cbk_rate_pct'],
       color='steelblue', edgecolor='white')
ax.axhline(cbk_rate, color='red', linestyle='--', linewidth=1.5, label=f'Promedio general ({cbk_rate:.1f}%)')
ax.set_xlabel('Número de Cuotas')
ax.set_ylabel('CBK Rate (%)')
ax.set_title('CBK Rate por Número de Cuotas de Pago', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('img/cbk_rate_cuotas.png')
plt.show()

print(cuotas_cbk[['payment_installments', 'cbk_rate_pct', 'total']].to_string(index=False))

## 6. Gráfico 3 — Distribución de Precio: CBK vs No-CBK

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=df[df['price'] < df['price'].quantile(0.95)],  # excluir outliers extremos
    x='cbk', y='price', ax=ax,
    palette={0: 'cornflowerblue', 1: 'tomato'}
)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Sin CBK', 'Con CBK'])
ax.set_ylabel('Precio ($)')
ax.set_title('Distribución de Precio por Presencia de CBK\n(sin outliers extremos, p95)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('img/precio_cbk_boxplot.png')
plt.show()

## 7. Gráfico 4 — Mapa de Calor de Correlaciones con CBK

In [ ]:
cols_corr = ['cbk', 'price', 'payment_installments', 'freight_value', 'discount', 'profit', 'has_shipping']
corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, linewidths=0.5)
ax.set_title('Correlación entre Variables y CBK', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('img/correlacion_cbk.png')
plt.show()

## 8. Gráfico 5 — Impacto del CBK en el Profit Total

In [ ]:
profit_sin_cbk = df[df['cbk'] == 0]['profit'].sum()
perdida_cbk_total = perdida_total  # ya calculado arriba
profit_neto = profit_sin_cbk + perdida_cbk_total

labels = ['Profit (sin CBK)', 'Pérdida por CBK', 'Profit Neto Total']
valores = [profit_sin_cbk, perdida_cbk_total, profit_neto]
colores = ['seagreen', 'tomato', 'steelblue']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, valores, color=colores, edgecolor='white')
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + abs(min(valores))*0.02,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('USD')
ax.set_title('Impacto del CBK en el Profit Total de Olist', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('img/impacto_cbk_profit.png')
plt.show()

## 9. Exportar CSV para SQL y Tableau

In [ ]:
cols_export = [
    'order_id', 'product_category_name', 'payment_installments', 'price',
    'overhead_cost', 'freight_value', 'cbk', 'profit', 'seller_id',
    'discount', 'has_shipping', 'tpv', 'cogs'
]
df[cols_export].to_csv('exports_tableau/export_para_sql.csv', index=False)
print(f'Exportado: exports_tableau/export_para_sql.csv ({len(df):,} filas)')

## 10. Resumen de Hallazgos (para el Deck)

In [ ]:
cat_top = cat_cbk.iloc[0]
cuotas_max = cuotas_cbk.loc[cuotas_cbk['cbk_rate_pct'].idxmax()]

print('=== HALLAZGOS PARA EL EXECUTIVE SUMMARY ===')
print(f'1. CBK rate general:           {cbk_rate:.2f}% ({total_cbk:,} órdenes de {total_ordenes:,})')
print(f'2. Pérdida total por CBK:      ${perdida_total:,.2f}')
print(f'3. Pérdida promedio por CBK:   ${perdida_promedio:,.2f}')
print(f'4. Categoría con más CBK:      {cat_top["product_category_name"]} ({cat_top["cbk_rate_pct"]:.1f}%)')
print(f'5. Cuotas con más CBK:         {int(cuotas_max["payment_installments"])}x ({cuotas_max["cbk_rate_pct"]:.1f}%)')
print(f'6. Impacto reducción CBK 50%:  ahorro de ${abs(perdida_total)*0.5:,.2f}')